# Notebook 14 — Ciclo completo com uma `pandas.Series`

**Público:** quem já conhece o vocabulário básico de séries temporais e quer operar o motor
completo sem esconder os critérios estatísticos.

**Pré-requisitos:** uma `pandas.Series` univariada, numérica, ordenada por um
`DatetimeIndex`. O notebook adiciona UTC quando o índice não possui timezone, mas não inventa
observações para gaps.

**Ao final você será capaz de:**

1. validar o contrato temporal;
2. caracterizar a série e ler as evidências;
3. declarar exatamente quais modelos serão avaliados;
4. selecionar por rolling-origin, cobertura e CRPS;
5. examinar resíduos, forecast e risco;
6. revelar observações futuras uma a uma e monitorar drift sem vazamento.

> A única célula obrigatoriamente editável está marcada com **EDITE SOMENTE ESTA CÉLULA**.
> As demais podem ser executadas de cima para baixo com **Run → Run All Cells**.


## 1. Mapa causal

O histórico será dividido antes do ajuste:

$$\text{desenvolvimento}\;|\;\text{cauda reservada para monitoramento}.$$

A parte de desenvolvimento passa por caracterização, backtest e seleção. A cauda não participa
dessas decisões: ela é revelada somente depois que o modelo e a política estão congelados.

```text
carregar → validar → separar → caracterizar → candidatos → backtest → selecionar
                                                         ↓
monitorar ← revelar cauda ← emitir forecast ← risco/simulação ← diagnosticar
```


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from dataclasses import asdict
from examples.memory_repository import MemoryRepository
from time_series_lab import (
    AR1Spec, ARIMASpec, BacktestConfig, ForecastEngine, GARCH11Spec,
    LocalLevelSpec, MonitoringPolicy, Observation, SelectionPolicy,
    ThresholdRisk, TimeSeries, TwoRegimeGaussianSpec, ValidationError,
)

%matplotlib inline
COLORS = {
    "blue": "#2563EB", "orange": "#EA580C", "green": "#059669",
    "purple": "#7C3AED", "gray": "#64748B", "red": "#DC2626",
}
plt.rcParams.update({
    "figure.figsize": (10, 4.8), "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.20,
    "axes.titleweight": "bold", "font.size": 11,
})
print("Ambiente pronto — NumPy", np.__version__)


## 2. Carregue sua série

Substitua o corpo de `load_user_series()` pelo código que lê seu CSV, banco ou API. O retorno
deve ser uma `pd.Series`; seu índice representa o tempo e seus valores representam uma única
variável.

O exemplo padrão simula um AR(1) com um pequeno deslocamento apenas na cauda. Assim o notebook
continua reproduzível antes de você conectar seus próprios dados.


In [ ]:
# ==================== EDITE SOMENTE ESTA CÉLULA ====================
def load_user_series() -> pd.Series:
    # Exemplo para CSV:
    # frame = pd.read_csv("meus_dados.csv", parse_dates=["timestamp"])
    # return frame.set_index("timestamp")["valor"].sort_index()

    rng = np.random.default_rng(1401)
    index = pd.date_range("2024-01-01", periods=240, freq="D", tz="UTC")
    values = np.empty(len(index)); values[0] = 0.0
    for t in range(1, len(index)):
        late_shift = 0.75 if t >= 230 else 0.0
        values[t] = 0.15 + late_shift + 0.72 * values[t-1] + rng.normal(0, 0.65)
    return pd.Series(values, index=index, name="valor")

USER_SERIES = load_user_series()
if not isinstance(USER_SERIES, pd.Series):
    raise TypeError("load_user_series() deve retornar uma pandas.Series")
USER_SERIES.head()


## 3. Configuração do experimento

- `MONITORING_HOLDOUT` define quantas observações finais ficam invisíveis durante a escolha.
- `SERIES_KIND="returns"` autoriza GARCH; em níveis genéricos ele não entra automaticamente.
- Markov switching é opcional porque seu ajuste em cada origem custa mais e exige uma hipótese
  substantiva de regimes.
- O fallback de cobertura é explícito: se nenhum candidato passar o guardrail, podemos parar ou
  selecionar apenas por CRPS, registrando que o guardrail foi relaxado.


In [ ]:
SERIES_ID = "serie-usuario"
FREQUENCY = None             # exemplo: "D"; None tenta inferir
TIMEZONE = "UTC"            # usado apenas se o índice for ingênuo
SERIES_KIND = "level"       # "level" ou "returns"
MONITORING_HOLDOUT = 10
FORECAST_HORIZON = 12
N_SIMULATION_PATHS = 5_000
RISK_LIMIT = None            # None usa média + 2 desvios do desenvolvimento
INCLUDE_MARKOV = False
ALLOW_COVERAGE_FALLBACK = True
RANDOM_SEED = 1401

if SERIES_KIND not in {"level", "returns"}:
    raise ValueError("SERIES_KIND deve ser 'level' ou 'returns'")


## 4. Contrato temporal e separação

A validação ocorre antes de qualquer teste estatístico. Duplicações, desordem, gaps, infinitos e
frequência incompatível interrompem o fluxo, pois ACF, horizonte e forecast dependem de uma
noção temporal coerente.

Exigimos ao menos 80 observações e deixamos pelo menos 60 no desenvolvimento. Esses números não
são leis estatísticas: são mínimos operacionais para que o exemplo tenha origens de backtest e
resíduos suficientes.


In [ ]:
raw = USER_SERIES.copy().sort_index()
if not isinstance(raw.index, pd.DatetimeIndex):
    raise TypeError("USER_SERIES.index deve ser um DatetimeIndex")
if raw.index.tz is None:
    raw.index = raw.index.tz_localize(TIMEZONE)
raw = raw.astype(float)
if len(raw) < 80:
    raise ValueError("Use ao menos 80 observações para o ciclo completo")
if not 1 <= MONITORING_HOLDOUT <= len(raw) - 60:
    raise ValueError("MONITORING_HOLDOUT deve preservar ao menos 60 observações de desenvolvimento")

development_pd = raw.iloc[:-MONITORING_HOLDOUT]
monitoring_pd = raw.iloc[-MONITORING_HOLDOUT:]
development = TimeSeries.from_pandas(
    SERIES_ID, development_pd, frequency=FREQUENCY, timezone_name=TIMEZONE,
    metadata={"series_kind": SERIES_KIND, "source": "notebook-14"},
)

repository = MemoryRepository()
engine = ForecastEngine(repository=repository)
engine.register_series(development)

pd.DataFrame({
    "parte": ["desenvolvimento", "monitoramento reservado"],
    "observações": [len(development_pd), len(monitoring_pd)],
    "início": [development_pd.index[0], monitoring_pd.index[0]],
    "fim": [development_pd.index[-1], monitoring_pd.index[-1]],
})


In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(raw.index, raw.values, color="black", lw=1, label="série carregada")
ax.axvspan(monitoring_pd.index[0], monitoring_pd.index[-1], color=COLORS["orange"],
           alpha=.18, label="cauda ainda não revelada")
ax.axvline(development_pd.index[-1], color=COLORS["red"], ls="--", label="cutoff")
ax.set_title("Separação definida antes da modelagem"); ax.legend(); plt.show()


## 5. Caracterização: evidências, não decisão final

A caracterização combina nível, diferenças, ACF/PACF, distribuição, tendência, ADF, KPSS,
Ljung–Box, ARCH-LM e um sinal descritivo de quebra. `suggested_models` indica famílias que
merecem entrar na comparação; não escolhe ordens nem promove modelos.

Como ler os principais testes:

- **ADF:** $H_0$ afirma raiz unitária; $p$ pequeno é evidência contra não estacionariedade;
- **KPSS:** $H_0$ afirma estacionariedade; $p$ pequeno é evidência contra estacionariedade;
- **Ljung–Box:** $H_0$ afirma ausência conjunta de autocorrelação até a defasagem testada;
- **ARCH-LM:** $H_0$ afirma ausência de dependência na variância condicional.

ACF e PACF são estimativas amostrais sujeitas a incerteza. Elas descrevem memória linear, mas
não provam sozinhas qual processo gerou os dados. ADF e KPSS têm hipóteses nulas opostas;
interpretá-los juntos é mais informativo do que tratar um único $p$-valor como diagnóstico.


In [ ]:
profile = engine.characterize(SERIES_ID, max_lag=24)
summary = pd.Series(profile.summary, name="valor").to_frame()
evidence = pd.DataFrame([asdict(item) for item in profile.evidence])
display(summary.round(4), evidence, pd.Series(profile.suggested_models, name="famílias sugeridas"))


In [ ]:
engine.plot_characterization(SERIES_ID, max_lag=24)
plt.show()


## 6. Candidatos realmente avaliados

A lista abaixo é a fonte de verdade do backtest. Incluímos candidatos simples e interpretáveis.
GARCH entra apenas quando `SERIES_KIND="returns"`; Markov switching só entra quando habilitado.

A caracterização poderia orientar uma busca maior, mas expandir muitas ordens sem separar uma
janela de pesquisa aumenta o risco de escolher por acaso.

As famílias correspondem às formulações:

$$\text{AR(1):}\quad Y_t=c+\phi Y_{t-1}+\varepsilon_t,$$

$$\text{ARIMA}(p,d,q):\quad
  \phi(B)(1-B)^dY_t=c+\theta(B)\varepsilon_t,$$

$$\text{nível local:}\quad
  Y_t=\mu_t+\varepsilon_t,\qquad \mu_t=\mu_{t-1}+\eta_t,$$

$$\text{GARCH(1,1):}\quad
  \sigma_t^2=\omega+\alpha\varepsilon_{t-1}^2+\beta\sigma_{t-1}^2.$$

Em Markov switching, os parâmetros dependem de um estado latente $S_t$ cuja evolução obedece
a $P(S_t=j\mid S_{t-1}=i)=p_{ij}$. Cada equação introduz uma hipótese diferente sobre memória,
tendência, variância ou regimes; por isso todas precisam ser julgadas fora da amostra.


In [ ]:
candidates = [
    AR1Spec(),
    ARIMASpec(order=(1, 0, 0), simulation_paths=300),
    ARIMASpec(order=(1, 0, 1), simulation_paths=300),
    LocalLevelSpec(),
]
if profile.adf_pvalue >= 0.05 or profile.kpss_pvalue <= 0.05:
    candidates.append(ARIMASpec(order=(1, 1, 1), simulation_paths=300))
else:
    candidates.append(ARIMASpec(order=(2, 0, 0), simulation_paths=300))
if SERIES_KIND == "returns":
    candidates.append(GARCH11Spec(mean="constant", simulation_paths=500))
if INCLUDE_MARKOV:
    candidates.append(TwoRegimeGaussianSpec(max_iterations=200))

candidate_labels = {
    f"{number:02d}": repr(spec) for number, spec in enumerate(candidates, start=1)
}
pd.Series(candidate_labels, name="especificação avaliada").to_frame()


## 7. Backtest rolling-origin

Em cada origem, todos os candidatos recebem o mesmo histórico anterior ao cutoff e são
avaliados nos mesmos realizados. O horizonte e o passo são escolhidos em função do tamanho do
desenvolvimento para que o notebook permaneça executável em séries de escalas diferentes.

Métricas:

- MAE/RMSE avaliam o centro da distribuição;
- cobertura e largura avaliam calibração e concentração do intervalo;
- CRPS avalia a distribuição inteira e é a métrica primária.

Para erros $e_i=y_i-\hat y_i$,

$$\operatorname{MAE}=\frac1n\sum_i|e_i|,\qquad
  \operatorname{RMSE}=\sqrt{\frac1n\sum_i e_i^2}.$$

A cobertura observada é a fração de realizados dentro de $[L_i,U_i]$. Já o CRPS compara toda
a CDF preditiva $F_i$ com o realizado:

$$\operatorname{CRPS}(F_i,y_i)=
  \int_{-\infty}^{\infty}\left(F_i(z)-\mathbf{1}\{y_i\le z\}\right)^2\,dz.$$

Menor CRPS é melhor, mas somente entre candidatos que respeitem os guardrails definidos.


In [ ]:
n_development = len(development_pd)
backtest_horizon = min(6, max(1, n_development // 30))
initial_train_size = max(40, int(0.65 * n_development))
backtest_step = max(1, backtest_horizon)
backtest_config = BacktestConfig(
    initial_train_size=initial_train_size,
    horizon=backtest_horizon,
    step=backtest_step,
    confidence=0.95,
    seed=RANDOM_SEED,
)
report = engine.backtest(SERIES_ID, candidates, backtest_config)

label_by_id = {
    score.model_run_family: repr(spec)
    for score, spec in zip(report.scores, report.candidates)
}
scoreboard = pd.DataFrame({
    label_by_id[score.model_run_family]: score.metrics for score in report.scores
}).T.sort_values("crps")
scoreboard.round(4)


In [ ]:
rows = pd.DataFrame(report.rows)
rows["candidate"] = rows["candidate_id"].map(label_by_id)
# Cobertura por horizonte é calculada explicitamente para não esconder o intervalo usado.
coverage_by_horizon = rows.assign(
    covered=lambda frame: (frame.actual >= frame.lower) & (frame.actual <= frame.upper)
).pivot_table(index="candidate", columns="horizon", values="covered", aggfunc="mean")
coverage_by_horizon.round(3)


## 8. Seleção e promoção

A política padrão exige cobertura agregada entre 80% e 100% e escolhe o menor CRPS entre os
elegíveis. Se ninguém passar, o notebook só relaxa o guardrail quando
`ALLOW_COVERAGE_FALLBACK=True`; o resultado registra essa decisão para não confundir fallback
com aprovação estatística.


In [ ]:
selection_policy = SelectionPolicy(primary_metric="crps", coverage_range=(0.80, 1.00))
coverage_fallback_used = False
try:
    decision = engine.select(report, selection_policy)
except ValidationError as error:
    if not ALLOW_COVERAGE_FALLBACK:
        raise
    coverage_fallback_used = True
    print("ATENÇÃO:", error)
    print("Fallback autorizado: seleção por CRPS sem guardrail de cobertura.")
    selection_policy = SelectionPolicy(primary_metric="crps", coverage_range=None)
    decision = engine.select(report, selection_policy)

promotion = engine.promote(
    SERIES_ID, decision.model_run_id, decision_id=decision.decision_id
)
selected_run = engine.get_champion(SERIES_ID)
pd.Series({
    "modelo selecionado": repr(decision.selected_spec),
    "métrica primária": decision.primary_metric,
    "CRPS": decision.scores["crps"],
    "cobertura": decision.scores["coverage"],
    "fallback de cobertura": coverage_fallback_used,
    "model_run_id": selected_run.model_run_id,
})


## 9. Diagnóstico residual

Seleção fora da amostra e diagnóstico residual respondem a perguntas diferentes. O backtest
pergunta “quem prevê melhor?”. Os resíduos perguntam “que estrutura o vencedor ainda deixou
para trás?”. Ljung–Box, memória nos quadrados, ARCH-LM e Q–Q não são guardrails automáticos
nesta versão; eles fundamentam revisão do conjunto de candidatos.


In [ ]:
diagnostics = engine.diagnose(SERIES_ID, model_ref=selected_run.model_run_id, max_lag=24)
display(pd.Series(diagnostics.tests, name="valor").round(4).to_frame())
for interpretation in diagnostics.interpretation:
    print("•", interpretation)
diagnostics.to_frame().tail()


In [ ]:
engine.plot_residuals(SERIES_ID, model_ref=selected_run.model_run_id, max_lag=24)
plt.show()


## 10. Forecast probabilístico

O resultado não é apenas uma linha. `PredictiveDistribution` oferece média, variância,
quantis, CDF e trajetórias. A faixa abaixo usa quantis de 2,5% e 97,5%; portanto ela não precisa
assumir que toda distribuição é Normal.

No caso AR(1), a previsão de um passo é aplicada recursivamente. Para
$Y_t=c+\phi Y_{t-1}+\varepsilon_t$, com
$\varepsilon_t\sim\mathcal N(0,\sigma^2)$,

$$\mathbb E_T[Y_{T+h}]
  =\phi^hY_T+c\frac{1-\phi^h}{1-\phi},$$

$$\operatorname{Var}_T(Y_{T+h})
  =\sigma^2\sum_{j=0}^{h-1}\phi^{2j}.$$

O primeiro termo da média é a memória do último valor; o segundo converge para a média de
longo prazo $c/(1-\phi)$ quando $|\phi|<1$. A variância acumula os choques futuros, ponderados
pela persistência. Para $\phi=1$, usa-se o limite: média $Y_T+ch$ e variância $h\sigma^2$.
Outros modelos implementam a mesma interface, embora sua distribuição seja calculada por
recorrências de estado, mistura ou simulação.


In [ ]:
forecast = engine.forecast(
    SERIES_ID, horizon=FORECAST_HORIZON, model_ref=selected_run.model_run_id,
    seed=RANDOM_SEED,
)
quantiles = forecast.distribution.quantile([0.025, 0.50, 0.975])
forecast_frame = pd.DataFrame({
    "mean": forecast.distribution.mean,
    "q025": quantiles[0],
    "median": quantiles[1],
    "q975": quantiles[2],
}, index=forecast.targets)
forecast_frame


In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
tail = development_pd.iloc[-min(80, len(development_pd)):]
ax.plot(tail.index, tail.values, color="black", label="histórico")
ax.plot(forecast_frame.index, forecast_frame["median"], color=COLORS["blue"], label="mediana")
ax.fill_between(forecast_frame.index, forecast_frame.q025, forecast_frame.q975,
                color=COLORS["blue"], alpha=.18, label="intervalo 95%")
ax.axvline(development_pd.index[-1], color=COLORS["red"], ls="--", label="cutoff")
ax.set_title("Forecast emitido antes de revelar a cauda"); ax.legend(); plt.show()


## 11. Simulação e risco de trajetória

O risco marginal $P(Y_{T+h}>L)$ é diferente do evento conjunto
$P(\max_hY_{T+h}>L)$. Somar probabilidades marginais contaria a mesma trajetória várias vezes.
Por isso o segundo evento é estimado sobre caminhos conjuntos.


In [ ]:
simulation = engine.simulate(
    SERIES_ID, horizon=FORECAST_HORIZON, n_paths=N_SIMULATION_PATHS,
    model_ref=selected_run.model_run_id, seed=RANDOM_SEED,
)
risk_limit = (
    float(RISK_LIMIT) if RISK_LIMIT is not None
    else profile.summary["mean"] + 2 * profile.summary["std"]
)
risk = engine.assess_risk(
    simulation, ThresholdRisk(limit=risk_limit, direction="above", seed=RANDOM_SEED)
)
risk_frame = pd.DataFrame({
    "marginal": risk.marginal_probability,
    "primeira_passagem": risk.first_passage_probability,
}, index=simulation.targets)
display(pd.Series({
    "limite": risk_limit,
    "probabilidade em qualquer horizonte": risk.any_horizon_probability,
}).round(4), risk_frame.round(4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for path in simulation.paths[:80]:
    axes[0].plot(simulation.targets, path, color=COLORS["blue"], alpha=.06)
axes[0].axhline(risk_limit, color=COLORS["red"], ls="--", label="limite")
axes[0].set_title("Amostra de trajetórias futuras"); axes[0].legend()
axes[1].plot(risk_frame.index, risk_frame.marginal, marker="o", label="marginal")
axes[1].plot(risk_frame.index, risk_frame.primeira_passagem, marker="o", label="1ª passagem")
axes[1].set_ylim(0, 1); axes[1].set_title("Probabilidades por horizonte"); axes[1].legend()
plt.tight_layout(); plt.show()


## 12. Change-point retrospectivo

A segmentação abaixo usa todo o desenvolvimento e, portanto, é retrospectiva. Ela ajuda a
investigar quebras, mas não pode ser apresentada como um alerta que existia antes dos dados.
A penalidade escala com a variância e com $\log(n)$ para evitar comparar SSEs em unidades
incompatíveis.


In [ ]:
change_penalty = 2 * np.var(development.values, ddof=1) * np.log(len(development.values))
change_report = engine.detect_changes(
    SERIES_ID, penalty=float(change_penalty), min_size=max(10, len(development.values)//20)
)
print("Quebras retrospectivas:", [str(value) for value in change_report.breakpoints])

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(development.timestamps, development.values, color="black", lw=1)
for point in change_report.breakpoints:
    ax.axvline(point, color=COLORS["red"], ls="--", alpha=.8)
ax.set_title("Change-points estimados usando todo o desenvolvimento"); plt.show()


## 13. Monitoramento prequential na cauda reservada

Agora congelamos a política e iniciamos a sessão. O motor já emite o forecast do próximo
instante. Para cada ponto reservado ele primeiro recupera essa distribuição, calcula a surpresa
e atualiza os detectores; só depois anexa o realizado, reajusta o mesmo modelo e emite o próximo
forecast.

A surpresa usa $\Phi^{-1}(F_t(y_t))$. Para Normal isso coincide com o resíduo padronizado; para
mistura de regimes usa a CDF completa.


In [ ]:
monitoring_policy = MonitoringPolicy(
    point_threshold=3.0,
    cusum_reference=0.4,
    cusum_threshold=6.0,
    ewma_smoothing=0.15,
    ewma_limit_multiplier=3.6,
    cooldown=3,
    reset_on_alert=True,
    refit_each_observation=True,
    forecast_horizon=1,
)
session = engine.start_monitoring(SERIES_ID, monitoring_policy)
monitoring_rows = []
last_cycle = None
for number, (timestamp, value) in enumerate(monitoring_pd.items(), start=1):
    last_cycle = engine.process_observation(
        session.session_id,
        Observation(timestamp=timestamp, value=float(value),
                    observation_id=f"notebook-14-{number:03d}"),
    )
    event = last_cycle.monitoring_event
    monitoring_rows.append({
        "timestamp": timestamp,
        "actual": event.actual,
        "surprise": event.surprise,
        "cusum_positive": event.cusum_positive,
        "cusum_negative": event.cusum_negative,
        "ewma": event.ewma,
        "ewma_limit": event.ewma_limit,
        "reasons": event.reasons,
        "alert_emitted": event.emitted,
        "forecast_id": event.forecast_id,
    })
monitoring = pd.DataFrame(monitoring_rows).set_index("timestamp")
monitoring


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
axes[0].plot(monitoring.index, monitoring.actual, color="black")
axes[0].set_title("Realizados revelados sequencialmente")
axes[1].plot(monitoring.index, monitoring.surprise, color=COLORS["blue"], marker="o")
axes[1].axhline(0, color="black", lw=.8); axes[1].axhline(3, color=COLORS["red"], ls="--")
axes[1].axhline(-3, color=COLORS["red"], ls="--"); axes[1].set_title("Surpresa prequential")
axes[2].plot(monitoring.index, monitoring.cusum_positive, label="CUSUM+")
axes[2].plot(monitoring.index, monitoring.cusum_negative, label="CUSUM−")
axes[2].plot(monitoring.index, monitoring.ewma, label="EWMA")
alerts = monitoring[monitoring.alert_emitted]
axes[2].scatter(alerts.index, alerts.ewma, color=COLORS["red"], label="alerta", zorder=5)
axes[2].set_title("Estatísticas sequenciais"); axes[2].legend()
plt.tight_layout(); plt.show()


## 14. Auditoria e resultado consolidado

Uma execução é confiável quando preserva relações causais, não apenas quando produz figuras.
Verificamos que cada realizado reservado encontrou um forecast anterior, que a série terminou
com o tamanho original e que o próximo forecast usa como cutoff o último realizado revelado.


In [ ]:
final_series = engine.get_series(SERIES_ID)
audit = {
    "observações carregadas": len(raw),
    "observações finais no motor": len(final_series.values),
    "eventos monitorados": len(monitoring),
    "eventos com forecast anterior": int(monitoring.forecast_id.notna().sum()),
    "alertas emitidos": int(monitoring.alert_emitted.sum()),
    "modelo selecionado": repr(decision.selected_spec),
    "fallback de cobertura": coverage_fallback_used,
    "CRPS selecionado": decision.scores["crps"],
    "risco conjunto": risk.any_horizon_probability,
    "change-points retrospectivos": len(change_report.breakpoints),
}
display(pd.Series(audit))

assert len(final_series.values) == len(raw)
assert monitoring.forecast_id.notna().all()
assert last_cycle is not None
assert last_cycle.next_forecast.cutoff == monitoring_pd.index[-1]
assert forecast.cutoff == development_pd.index[-1]
print("Testes rápidos do Notebook 14 passaram.")


## 15. Como interpretar sem automatizar demais

- Uma família sugerida pela caracterização é apenas candidata.
- O menor CRPS não compensa cobertura inaceitável quando o guardrail está ativo.
- Resíduos autocorrelacionados indicam dinâmica faltante; resíduos quadráticos correlacionados
  indicam variância faltante.
- Change-point retrospectivo não antecipa uma quebra.
- Um alerta é evidência para investigação, não autorização automática para intervenção.
- Se os dados forem retornos, configure `SERIES_KIND="returns"`; não aplique GARCH a níveis
  apenas porque o adaptador existe.

**Extensões:** testar outras ordens, comparar métricas por horizonte, calibrar limiares por ARL,
implementar repositório transacional e executar várias séries em processos separados.


## 16. Exercícios

1. Adicione `ARIMASpec(order=(2,1,2))` e compare CRPS, cobertura e resíduos.
2. Desative `ALLOW_COVERAGE_FALLBACK` e observe a falha quando nenhum modelo é elegível.
3. Use uma série de retornos, configure `SERIES_KIND="returns"` e avalie GARCH.
4. Habilite `INCLUDE_MARKOV` somente depois de formular uma hipótese de regimes.
5. Altere `RISK_LIMIT` e compare risco marginal, qualquer horizonte e primeira passagem.
6. Aumente a cauda reservada e compare falso alarme, atraso e custo computacional do refit.

<details><summary><strong>Gabarito orientativo</strong></summary>

1. Acrescente o spec à lista; mais parâmetros não garantem melhor previsão fora da amostra.
2. O erro “nenhum candidato passou os guardrails” é o comportamento seguro esperado.
3. Verifique ACF dos quadrados e ARCH-LM antes de interpretar o GARCH.
4. Compare probabilidades de regime, não apenas a classificação mais provável.
5. A probabilidade de qualquer horizonte não é a soma das marginais.
6. Uma cauda maior mede melhor o monitor, mas deixa menos dados para seleção.

</details>


In [ ]:
# Scaffold executável para o exercício 1:
exercise_candidates = [*candidates, ARIMASpec(order=(2, 1, 2), simulation_paths=300)]
print("Candidatos atuais:", len(candidates))
print("Candidatos após a extensão:", len(exercise_candidates))
# Para executar a comparação, troque `candidates` por `exercise_candidates`
# na célula de backtest e execute novamente a partir daquela seção.


## Conclusão

Uma única `pandas.Series` percorreu o sistema inteiro com separação causal entre pesquisa e
monitoramento. O notebook deixou visíveis os candidatos, os critérios, os resíduos, a
distribuição futura, o risco e os eventos sequenciais. Para produção, substitua o repositório
em memória e mantenha os mesmos contratos estatísticos e de auditoria.
